In [0]:
import pandas as pd

model_data_dir = "dbfs:/Volumes/lsg/pre_dm/tfsdl_lsg_ops_prod/inventory_rebalance/model_data"
lane_candidates_model_df = spark.read.format("parquet").load(model_data_dir)
pdf = lane_candidates_model_df.toPandas().copy()

numeric_cols = [
    "feasible_move_qty_row",
    "max_available_to_move",
    "max_available_to_receive",
    "days_to_dnsa",
    "source_candidate_score",
    "destination_candidate_score_adjusted",
]

for c in numeric_cols:
    pdf[c] = pd.to_numeric(pdf[c], errors="coerce").fillna(0)

pdf = pdf.sort_values(
    by=[
        "sku_number",
        "sku_source",
        "days_to_dnsa",
        "source_candidate_score",
        "destination_candidate_score_adjusted",
    ],
    ascending=[True, True, False, False, False]
).reset_index(drop=True)

lot_caps = (
    pdf[
        [
            "sku_number",
            "sku_source",
            "source_branch",
            "lot_number",
            "inventory_location_code",
            "feasible_move_qty_row",
        ]
    ]
    .drop_duplicates()
    .set_index(
        [
            "sku_number",
            "sku_source",
            "source_branch",
            "lot_number",
            "inventory_location_code",
        ]
    )["feasible_move_qty_row"]
    .to_dict()
)

source_caps = (
    pdf[
        [
            "sku_number",
            "sku_source",
            "source_branch",
            "max_available_to_move",
        ]
    ]
    .drop_duplicates()
    .set_index(
        [
            "sku_number",
            "sku_source",
            "source_branch",
        ]
    )["max_available_to_move"]
    .to_dict()
)

destination_caps = (
    pdf[
        [
            "sku_number",
            "sku_source",
            "destination_branch",
            "max_available_to_receive",
        ]
    ]
    .drop_duplicates()
    .set_index(
        [
            "sku_number",
            "sku_source",
            "destination_branch",
        ]
    )["max_available_to_receive"]
    .to_dict()
)

allocations = []

for _, row in pdf.iterrows():
    lot_key = (
        row["sku_number"],
        row["sku_source"],
        row["source_branch"],
        row["lot_number"],
        row["inventory_location_code"],
    )

    source_key = (
        row["sku_number"],
        row["sku_source"],
        row["source_branch"],
    )

    destination_key = (
        row["sku_number"],
        row["sku_source"],
        row["destination_branch"],
    )

    remaining_lot = lot_caps.get(lot_key, 0)
    remaining_source = source_caps.get(source_key, 0)
    remaining_destination = destination_caps.get(destination_key, 0)

    alloc_qty = min(
        remaining_lot,
        remaining_source,
        remaining_destination,
    )

    allocations.append(alloc_qty)

    lot_caps[lot_key] = remaining_lot - alloc_qty
    source_caps[source_key] = remaining_source - alloc_qty
    destination_caps[destination_key] = remaining_destination - alloc_qty

pdf["optimized_move_qty"] = allocations
pdf = pdf[pdf["optimized_move_qty"] > 0].copy()

optimized_df = spark.createDataFrame(pdf)
display(optimized_df)


In [0]:

greedy_output = "dbfs:/Volumes/lsg/pre_dm/tfsdl_lsg_ops_prod/inventory_rebalance/greedy_output"
optimized_df.write.mode("overwrite").format("parquet").save(greedy_output)
